# NCA Training (Single or Multi-Texture)

This notebook trains a Neural Cellular Automata (NCA) model to produce textures.

**Single texture mode (num_textures=1):**
- Standard NCA training with one target texture
- Equivalent to the original `nca_training.ipynb`

**Multi-texture mode (num_textures=2 or 3):**
- Train a single model to produce 2 or 3 different textures
- Each texture has its own initial noise strength and runtime noise level
- The network learns to associate noise settings with texture output

In [ ]:
# @title Imports and Notebook Utilities
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Loss Function Selection and Definitions
import torch.nn.functional as F

#@markdown ### Loss Function Type
loss_type = "sliced_ot"  #@param ["sliced_ot", "relaxed_ot"]

print(f"Selected loss function: {loss_type}")

# Load VGG for feature extraction
vgg = models.vgg16(weights='IMAGENET1K_V1').features

# ============================================================================
# Sliced OT Loss 
# ============================================================================
def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_sliced_ot_loss(target_img):
    """Create a Sliced OT loss function for a target image."""
    with torch.no_grad():
        yy = calc_styles_vgg(target_img, vgg)
    def loss_f(imgs):
        xx = calc_styles_vgg(imgs, vgg)
        return sum(ot_loss(x, y) for x, y in zip(xx, yy))
    return loss_f

# ============================================================================
# Relaxed OT Loss 
# ============================================================================
class RelaxedOTLoss(torch.nn.Module):
    """https://arxiv.org/abs/1904.12785"""
    def __init__(self, target_image, n_samples=1024):
        super().__init__()
        self.n_samples = n_samples
        with torch.no_grad():
            self.target_features = self.get_vgg_features(target_image)

    def get_vgg_features(self, imgs):
        style_layers = [1, 6, 11, 18, 25]
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        x = (imgs - mean) / std
        b, c, h, w = x.shape
        features = []
        for i, layer in enumerate(vgg[:max(style_layers) + 1]):
            x = layer(x)
            if i in style_layers:
                b, c, h, w = x.shape
                features.append(x.reshape(b, c, h * w))
        return features

    @staticmethod
    def pairwise_distances_cos(x, y):
        x_norm = torch.norm(x, dim=2, keepdim=True)
        y_t = y.transpose(1, 2)
        y_norm = torch.norm(y_t, dim=1, keepdim=True)
        dist = 1. - torch.matmul(x, y_t) / (x_norm * y_norm + 1e-10)
        return dist

    @staticmethod
    def style_loss(x, y):
        pairwise_distance = RelaxedOTLoss.pairwise_distances_cos(x, y)
        m1, m1_inds = pairwise_distance.min(1)
        m2, m2_inds = pairwise_distance.min(2)
        remd = torch.max(m1.mean(dim=1), m2.mean(dim=1))
        return remd

    @staticmethod
    def moment_loss(x, y):
        mu_x, mu_y = torch.mean(x, 1, keepdim=True), torch.mean(y, 1, keepdim=True)
        mu_diff = torch.abs(mu_x - mu_y).mean(dim=(1, 2))
        x_c, y_c = x - mu_x, y - mu_y
        x_cov = torch.matmul(x_c.transpose(1, 2), x_c) / (x.shape[1] - 1)
        y_cov = torch.matmul(y_c.transpose(1, 2), y_c) / (y.shape[1] - 1)
        cov_diff = torch.abs(x_cov - y_cov).mean(dim=(1, 2))
        return mu_diff + cov_diff

    def forward(self, generated_image):
        loss = 0.0
        generated_features = self.get_vgg_features(generated_image)
        for x, y in zip(generated_features, self.target_features):
            (b_x, c, n_x), (b_y, _, n_y) = x.shape, y.shape
            n_samples = min(n_x, n_y, self.n_samples)
            indices_x = torch.argsort(torch.rand(b_x, 1, n_x, device=x.device), dim=-1)[..., :n_samples]
            x = x.gather(-1, indices_x.expand(b_x, c, n_samples))
            indices_y = torch.argsort(torch.rand(b_y, 1, n_y, device=y.device), dim=-1)[..., :n_samples]
            y = y.gather(-1, indices_y.expand(b_y, c, n_samples))
            x, y = x.transpose(1, 2), y.transpose(1, 2)
            loss += self.style_loss(x, y) + self.moment_loss(x, y)
        return loss.mean()

def create_loss_fn(target_img):
    """Create loss function based on selected type."""
    if loss_type == "sliced_ot":
        return create_sliced_ot_loss(target_img)
    elif loss_type == "relaxed_ot":
        return RelaxedOTLoss(target_img, n_samples=1024)
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

print(f"VGG loaded and {loss_type} loss function defined.")

In [ ]:
#@title Load Target Images
#@markdown ### Number of Textures
num_textures = 1  #@param [1, 2, 3] {type: "raw"}

from google.colab import files

target_imgs = []
texture_paths = []

for i in range(num_textures):
    if num_textures == 1:
        label = "target texture"
    elif i == 0:
        label = "Texture 1"
    elif i == 1 and num_textures == 2:
        label = "Texture 2"
    elif i == 1:
        label = "Texture 2"
    else:
        label = "Texture 3"

    print(f"\nUpload {label}:")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    texture_paths.append(path)

    img = imread(io.BytesIO(uploaded[path]), max_size=128)
    target_imgs.append(img)
    print(f"{label}:")
    imshow(img)

# Create loss functions for each target
loss_fns = []
for i, img in enumerate(target_imgs):
    tensor = torch.tensor(img).permute(2, 0, 1).unsqueeze(0)
    loss_fn = create_loss_fn(tensor)
    loss_fns.append(loss_fn)
    print(f"Created {loss_type} loss function for texture {i+1}")

print(f"\n\u2713 Loaded {num_textures} texture(s) with {loss_type} loss")

In [ ]:
#@title NoiseNCA Architecture

#@markdown ### Model Architecture
channel_n = 12  #@param {type: "integer"}

def depthwise_conv(x, filters):
    """filters: [filter_n, h, w]"""
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)

def merge_lap(z):
    # Merge lap_x and lap_y into a single laplacian filter
    b, c, h, w = z.shape  # [b, 5 * chn, h, w]
    z = torch.stack([
        z[:, ::5],
        z[:, 1::5],
        z[:, 2::5],
        z[:, 3::5] + z[:, 4::5]
    ], dim=2)  # [b, chn, 4, h, w]
    return z.reshape(b, -1, h, w)  # [b, 4 * chn, h, w]


class NoiseNCA(torch.nn.Module):
    def __init__(self, chn=12, fc_dim=96, noise_level=1.0):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
            lap_x = torch.tensor([[0.5, 0.0, 0.5], [2.0, -6.0, 2.0], [0.5, 0.0, 0.5]])
            self.filters = torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T])

    def perception(self, s, dx=1.0, dy=1.0):
        z = depthwise_conv(s, self.filters)  # [b, 5 * chn, h, w]
        if isinstance(dx, float) and dx == 1.0 and isinstance(dy, float) == 1.0:
            return merge_lap(z)

        if not isinstance(dx, torch.Tensor) or dx.ndim != 3:
            dx = torch.tensor([dx], device=s.device)[:, None, None]
        if not isinstance(dy, torch.Tensor) or dy.ndim != 3:
            dy = torch.tensor([dy], device=s.device)[:, None, None]

        scale = 1.0 / torch.stack([torch.ones_like(dx), dx, dy, dx ** 2, dy ** 2], dim=1)
        scale = torch.tile(scale, (1, self.chn, 1, 1))
        z = z * scale
        return merge_lap(z)

    def forward(self, s, dx=1.0, dy=1.0, dt=1.0, noise=None):
        if noise is not None:
            # Support both scalar and per-batch noise
            if isinstance(noise, torch.Tensor) and noise.ndim >= 1:
                # Per-batch noise: reshape to [b, 1, 1, 1] for broadcasting
                if noise.ndim == 1:
                    noise = noise.reshape(-1, 1, 1, 1)
            s = s + torch.randn_like(s) * noise
        z = self.perception(s, dx, dy)
        delta_s = self.w2(torch.relu(self.w1(z)))
        return s + delta_s * dt

    def seed(self, n, h=128, w=128, noise_level=None):
        """
        Create initial seed states.
        
        Args:
            n: Number of samples (or list of per-sample noise levels)
            h: Height
            w: Width
            noise_level: Noise level(s) for initial state. Can be:
                - None: use self.noise_level
                - float: use same noise for all samples
                - tensor/array of shape [n]: per-sample noise levels
        """
        # Handle case where n is actually a list/array of noise levels
        if isinstance(n, (list, np.ndarray, torch.Tensor)):
            noise_level = n
            n = len(noise_level)
        
        # Determine noise level
        if noise_level is None:
            nl = self.noise_level.item()
            return (torch.rand(n, self.chn, h, w) - 0.5) * nl
        elif isinstance(noise_level, (int, float)):
            return (torch.rand(n, self.chn, h, w) - 0.5) * noise_level
        else:
            # Per-sample noise levels: [n] -> [n, 1, 1, 1] for broadcasting
            if isinstance(noise_level, np.ndarray):
                noise_level = torch.tensor(noise_level, dtype=torch.float32)
            noise_level = noise_level.reshape(-1, 1, 1, 1)
            return (torch.rand(n, self.chn, h, w) - 0.5) * noise_level


def to_rgb(s):
    return s[..., :3, :, :] + 0.5


# Initialize model with selected channel count
model = NoiseNCA(chn=channel_n)
param_n = sum(p.numel() for p in model.parameters())
print(f'NoiseNCA with {channel_n} channels')
print(f'Parameter count: {param_n}')

In [ ]:
#@title Setup Training
import os
import glob
from google.colab import files

#@markdown ### Noise Settings Per Texture
#@markdown Set initial (seed) noise and runtime noise for each texture.
#@markdown - **Init noise**: Controls which texture emerges (all default to 1.0)
#@markdown - **Runtime noise**: Applied during forward steps

#@markdown **Texture 1:**
texture1_init_noise = 1.0  #@param {type: "number"}
texture1_runtime_noise = 0.0  #@param {type: "number"}

#@markdown **Texture 2 (only used if num_textures >= 2):**
texture2_init_noise = 1.0  #@param {type: "number"}
texture2_runtime_noise = 0.02  #@param {type: "number"}

#@markdown **Texture 3 (only used if num_textures = 3):**
texture3_init_noise = 1.0  #@param {type: "number"}
texture3_runtime_noise = 0.04  #@param {type: "number"}

# Build arrays based on num_textures
if num_textures == 1:
    seed_noise_levels = [texture1_init_noise]
    runtime_noise_levels = [texture1_runtime_noise]
elif num_textures == 2:
    seed_noise_levels = [texture1_init_noise, texture2_init_noise]
    runtime_noise_levels = [texture1_runtime_noise, texture2_runtime_noise]
else:
    seed_noise_levels = [texture1_init_noise, texture2_init_noise, texture3_init_noise]
    runtime_noise_levels = [texture1_runtime_noise, texture2_runtime_noise, texture3_runtime_noise]

print(f"Noise settings for {num_textures} texture(s):")
for i in range(num_textures):
    print(f"  Texture {i+1}: init_noise={seed_noise_levels[i]:.2f}, runtime_noise={runtime_noise_levels[i]:.3f}")

# Check for existing checkpoints
checkpoint_files = glob.glob('nca_*.pt') + glob.glob('noise_controlled_*.pt') + glob.glob('checkpoint_*.pt')
checkpoint_files = list(set(checkpoint_files))  # Remove duplicates

if checkpoint_files:
    print(f"\nFound {len(checkpoint_files)} checkpoint file(s):")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")
    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("\nNo checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()
    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Use model from architecture cell (already initialized with channel_n)
if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iter = checkpoint.get('iteration', 0) + 1
    loss_log = checkpoint.get('loss_log', [])
    pool = checkpoint.get('pool', None)
    
    # Restore noise levels from checkpoint if available
    if 'seed_noise_levels' in checkpoint:
        saved_seed = checkpoint['seed_noise_levels']
        saved_runtime = checkpoint.get('runtime_noise_levels', [0.0] * len(saved_seed))
        print(f"\nCheckpoint had: seed_noise={saved_seed}, runtime_noise={saved_runtime}")
        print(f"Using current settings: seed_noise={seed_noise_levels}, runtime_noise={runtime_noise_levels}")
    
    print(f"Resumed from iteration {start_iter - 1}")
else:
    start_iter = 0
    loss_log = []
    pool = None
    print("Starting fresh training")

# Initialize pool with texture-specific seed noise levels
pool_size = 256
if pool is None:
    with torch.no_grad():
        # Each pool slot gets the seed noise level of its corresponding texture
        pool_seed_noise = np.array([seed_noise_levels[i % num_textures] for i in range(pool_size)])
        pool = model.seed(pool_seed_noise)
    print(f"Initialized pool with texture-specific seed noise levels")

# Optimizer with adaptive LR
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.3, patience=500,
    threshold=0.01, threshold_mode='rel', min_lr=1e-6
)

if checkpoint and 'optimizer_state_dict' in checkpoint:
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
if checkpoint and 'scheduler_state_dict' in checkpoint:
    lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])

print(f"\nPool shape: {pool.shape}")

In [ ]:
#@title Training Loop {vertical-output: true}

num_iterations = 10000  #@param {type: "integer"}
batch_size = 4  #@param {type: "integer"}

try:
    for i in range(start_iter, start_iter + num_iterations):
        with torch.no_grad():
            # Sample batch indices from pool
            batch_idx = np.random.choice(len(pool), batch_size, replace=False)
            s = pool[batch_idx]

            # Map batch indices to texture indices (0, 1, or 2)
            texture_indices = batch_idx % num_textures

            # Inject fresh seed periodically with texture-specific noise level
            if i % 32 == 0:
                seed_nl = seed_noise_levels[texture_indices[0]]
                s[:1] = model.seed(1, noise_level=seed_nl)

            # Get runtime noise level for each batch element
            batch_noise = torch.tensor(
                [runtime_noise_levels[t] for t in texture_indices],
                device='cuda', dtype=torch.float32
            )

        # Run forward steps with per-texture runtime noise
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, noise=batch_noise)

        # Compute loss
        overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()
        rgb = to_rgb(s)

        if num_textures == 1:
            # Single texture: simpler loss computation
            loss = overflow_loss + loss_fns[0](rgb)
        else:
            # Multi-texture: each batch element uses its corresponding target
            loss = overflow_loss
            for b in range(batch_size):
                t_idx = texture_indices[b]
                loss = loss + loss_fns[t_idx](rgb[b:b+1])

        # Backward and optimize
        with torch.no_grad():
            loss.backward()
            for p in model.parameters():
                p.grad /= (p.grad.norm() + 1e-8)
            opt.step()
            opt.zero_grad()
            lr_sched.step(loss)
            pool[batch_idx] = s.detach()
            loss_log.append(loss.item())

            # Display progress
            if i % 10 == 0:
                lr = opt.param_groups[0]['lr']
                display(Markdown(f"iter: {i}, loss: {loss.item():.2e}, lr: {lr:.2e}"), display_id='stats')

            # Visualize
            if i % 20 == 0:
                # Show loss plot and current batch
                pl.figure(figsize=(12, 3))
                
                pl.subplot(1, 2, 1)
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.title('Loss')
                pl.xlabel('Iteration')

                # Show current batch with texture labels
                pl.subplot(1, 2, 2)
                imgs = rgb.permute(0, 2, 3, 1).cpu().numpy()
                if num_textures == 1:
                    pl.title('Current batch')
                else:
                    labels = [f'T{t+1}' for t in texture_indices]
                    pl.title(' | '.join(labels))
                pl.imshow(np.hstack(imgs))
                pl.axis('off')

                pl.tight_layout()
                imshow(grab_plot(), id='progress')

            # Save checkpoint
            if i % 1000 == 0 and i > start_iter:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'scheduler_state_dict': lr_sched.state_dict(),
                    'loss_log': loss_log,
                    'pool': pool,
                    'seed_noise_levels': seed_noise_levels,
                    'runtime_noise_levels': runtime_noise_levels,
                    'num_textures': num_textures,
                }
                torch.save(checkpoint, f'nca_iter_{i}.pt')
                print(f"\nCheckpoint saved at iteration {i}")

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
        'seed_noise_levels': seed_noise_levels,
        'runtime_noise_levels': runtime_noise_levels,
        'num_textures': num_textures,
    }
    torch.save(checkpoint, f'nca_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved!')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Test Model

print("Testing model with trained noise settings:\n")

with torch.no_grad():
    # Test each texture with its specific init and runtime noise
    results = []
    for t in range(num_textures):
        s = model.seed(1, noise_level=seed_noise_levels[t])
        for _ in range(64):
            s = model(s, noise=runtime_noise_levels[t])
        results.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())

    # Display model outputs
    fig, axes = pl.subplots(1, num_textures, figsize=(4 * num_textures, 4))
    if num_textures == 1:
        axes = [axes]
    for t, (ax, img) in enumerate(zip(axes, results)):
        ax.imshow(np.clip(img, 0, 1))
        if num_textures == 1:
            ax.set_title(f'Model output\ninit={seed_noise_levels[t]:.2f}, runtime={runtime_noise_levels[t]:.3f}')
        else:
            ax.set_title(f'Texture {t+1}\ninit={seed_noise_levels[t]:.2f}, runtime={runtime_noise_levels[t]:.3f}')
        ax.axis('off')
    pl.suptitle('Model Output')
    pl.tight_layout()
    imshow(grab_plot())

# Show target textures for comparison
print("\nTarget texture(s) for reference:")
fig, axes = pl.subplots(1, num_textures, figsize=(4 * num_textures, 4))
if num_textures == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, target_imgs)):
    ax.imshow(img)
    if num_textures == 1:
        ax.set_title('Target')
    else:
        ax.set_title(f'Target {i+1}')
    ax.axis('off')
pl.tight_layout()
imshow(grab_plot())

# For multi-texture: sweep across noise levels
if num_textures > 1:
    # Sweep across init noise levels (with no runtime noise)
    print("\nSweep across init noise levels (runtime noise = 0):")
    min_test = min(seed_noise_levels) * 0.5
    max_test = max(seed_noise_levels) * 1.5
    test_init_levels = np.linspace(min_test, max_test, 8)

    with torch.no_grad():
        sweep_results = []
        for nl in test_init_levels:
            s = model.seed(1, noise_level=nl)
            for _ in range(64):
                s = model(s, noise=0.0)
            sweep_results.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())

        fig, axes = pl.subplots(1, len(test_init_levels), figsize=(16, 2))
        for ax, img, nl in zip(axes, sweep_results, test_init_levels):
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title(f'{nl:.2f}')
            ax.axis('off')
        pl.suptitle('Init Noise Sweep (no runtime noise)')
        pl.tight_layout()
        imshow(grab_plot())

    # Sweep across runtime noise levels (with fixed init noise)
    print("\nSweep across runtime noise levels (init noise = 1.0):")
    max_runtime = max(runtime_noise_levels) * 1.5 if max(runtime_noise_levels) > 0 else 0.06
    test_runtime_levels = np.linspace(0, max_runtime, 8)

    with torch.no_grad():
        sweep_results = []
        for nl in test_runtime_levels:
            s = model.seed(1, noise_level=1.0)
            for _ in range(64):
                s = model(s, noise=nl)
            sweep_results.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())

        fig, axes = pl.subplots(1, len(test_runtime_levels), figsize=(16, 2))
        for ax, img, nl in zip(axes, sweep_results, test_runtime_levels):
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title(f'{nl:.3f}')
            ax.axis('off')
        pl.suptitle('Runtime Noise Sweep (init noise = 1.0)')
        pl.tight_layout()
        imshow(grab_plot())

In [ ]:
#@title Save Model Weights

# Save for demo
weights_file = 'weights.pt'

# Include training params for conversion script
state_dict = model.state_dict()
state_dict['_training_params'] = {
    'seed_noise_levels': seed_noise_levels,
    'runtime_noise_levels': runtime_noise_levels,
    'num_textures': num_textures,
    'model_type': 'nca',
    'loss_type': loss_type,
    'texture_paths': texture_paths,
    'channel_n': channel_n,
}

torch.save(state_dict, weights_file)
print(f"Saved: {weights_file}")
print(f"  Channels: {channel_n}")
print(f"  Textures: {num_textures}")
print(f"  Loss type: {loss_type}")
if num_textures > 1:
    print(f"  Seed noise levels: {seed_noise_levels}")
    print(f"  Runtime noise levels: {runtime_noise_levels}")
else:
    print(f"  Init noise: {seed_noise_levels[0]}")
    print(f"  Runtime noise: {runtime_noise_levels[0]}")

# Also save full checkpoint
checkpoint_file = 'checkpoint_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
    'seed_noise_levels': seed_noise_levels,
    'runtime_noise_levels': runtime_noise_levels,
    'num_textures': num_textures,
    'channel_n': channel_n,
}
torch.save(checkpoint, checkpoint_file)
print(f"\nFull checkpoint saved: {checkpoint_file}")

# Download
from google.colab import files
files.download(weights_file)